In [3]:
import ipywidgets as widgets
from ipywidgets import AppLayout, Button, Layout, HBox, VBox, NaiveDatetimePicker
from IPython.display import display,IFrame
import pandas as pd
import matplotlib.pyplot as plt
from bokeh.plotting import figure
from bokeh.io import output_file, save
from bokeh.models import ColumnDataSource
import ast  # to safely parse list strings

output_file("bokeh_plot.html")

class Dashboard:
    def __init__(self, df):
        """
        Initialize the dashboard with a DataFrame.

        Args:
            df (pd.DataFrame): The DataFrame to display or interact with.
        """
        self.original_df = df  # Store the DataFrame

        self.min_time_from_df = pd.to_datetime(self.original_df['seen'].min())

        self.max_time_from_df = pd.to_datetime(self.original_df['seen'].max())

        self.filtered_df =  pd.DataFrame(columns=df.columns) # Store empty df but keep columns of original

        self.feilds_to_plot = self.get_feilds_to_plot(self.original_df)

        self.warning_flag = False

        self._create_ui()  # Create the UI widgets
        self._wire_up_events()  # Attach event handlers to widgets

    def _create_ui(self):
        """
        Create and layout all UI components for the dashboard.
        """
        # Dashboard title (centered)
        self.dashboard_title = widgets.HTML(
            value="<div style='text-align: center; width: 100%;'><h1>My Dashboard Title</h1></div>"
        )

        # Date/time pickers
        self.StartDatetimePicker = widgets.NaiveDatetimePicker(description='Pick a Time',value=self.min_time_from_df)
        self.EndDatetimePicker = widgets.NaiveDatetimePicker(description='Pick a Time',value=self.max_time_from_df)

        # Radio button selection for plot types
        self.plot_picker = widgets.RadioButtons(
            options=['A', 'B', 'C'],
            value='A',
            layout=widgets.Layout(justify_items='center', align_items='center')
        )

        # Selection dropdown for OS filter
        self.field_Select = widgets.Select(
            options=self.feilds_to_plot,
            value=None,
                description=' ',
            layout=widgets.Layout(justify_items='center', align_items='center')
        )

        # Arrange widgets into rows and columns
        # Each VBox contains a title (HTML) and corresponding control(s)
        user_options = HBox([
            VBox([
                widgets.HTML("<div style='text-align: center; width: 100%;'><h3>Plot</h3></div>"),
                self.plot_picker
            ]),
            VBox([
                widgets.HTML("<div style='text-align: center; width: 100%;'><h3>Start/End Time</h3></div>"),
                self.StartDatetimePicker,
                self.EndDatetimePicker
            ]),
            VBox([
                widgets.HTML("<div style='text-align: center; width: 100%;'><h3>Field to plot</h3></div>"),
                self.field_Select
            ])
        ])

        #generates widgets for advanced
        filter_options_output = self.filter_options()

        # Accordion for additional filter options
        self.filter_accordion = widgets.Accordion(children=[filter_options_output])

        self.filter_accordion.set_title(0, 'Filter Options')

        # Combine user options and accordion vertically
        user_options_with_filter_accordion = VBox([user_options, self.filter_accordion])

        # Buttons for triggering actions
        self.go_button = Button(
            description='Go',
            button_style='success',
            layout=Layout(height='50px', width='300px')
        )

        self.reset_button = Button(
            description='Reset',
            button_style='warning',
            layout=Layout(height='50px', width='300px')
        )

        # Place buttons horizontally at the bottom
        go_and_reset_buttons = HBox(
            [self.go_button, self.reset_button],
            layout=Layout(
                height='50px',
                display='flex',
                width='auto',
                justify_content='center'
            )
        )

        # Output widget for displaying DataFrame or plots
        self.plot_output = widgets.Output()

        # Assemble the complete UI using AppLayout
        self.ui = VBox([
            AppLayout(
                header=self.dashboard_title,
                left_sidebar=None,
                center=user_options_with_filter_accordion,
                right_sidebar=None,
                footer=go_and_reset_buttons,
                layout=Layout(border='3px solid black')
            ),
            self.plot_output
        ])

    def _wire_up_events(self):
        """
        Attach button click events to their respective callbacks.
        """
        self.go_button.on_click(self.on_go_click)
        self.reset_button.on_click(self.on_reset_click)

    def on_go_click(self, b):
        """
        Callback function for the "Go" button.

        Clears the previous output and displays the DataFrame.
        """
        self.warning_flag = False
        
        with self.plot_output:  
            self.plot_output.clear_output()
            
            if self.StartDatetimePicker.value is None or self.EndDatetimePicker.value is None:
                self.display_message_to_user("Please select both dates") 
            else:
                self.filter_df_by_date_time()
                
                if self.filtered_df.empty:
                    self.display_message_to_user("No events in range consider changing date parameters")
                # else:
                #     display(self.filtered_df)
                
            self.apply_user_filters()

            if self.filtered_df.empty:
                self.display_message_to_user("No events in range consider changing filter options") 



            if  self.field_Select.value is None and self.warning_flag == False:
                self.display_message_to_user("Please select a Field To Plot")
                
            elif self.warning_flag == False:

                if self.plot_picker.value == 'A':
                    self.plot_A()
                elif self.plot_picker.value == 'B':
                    self.plot_B()
                elif self.plot_picker.value == 'C':
                    self.plot_C()

    def on_reset_click(self, b):
        """
        Callback function for the "Reset" button.

        Clears the output area.
        """
        self.warning_flag = False
        
        with self.plot_output:
            self.plot_output.clear_output()
            self.StartDatetimePicker.value = self.min_time_from_df
            self.EndDatetimePicker.value = self.max_time_from_df
            self.plot_picker.value = 'A'
            self.field_Select.value = None
            self.filter_accordion.selected_index = None

            # done this way as it's a tuple
            children = list(self.filter_accordion.children)
            children[0] = self.filter_options()
            self.filter_accordion.children = tuple(children)


    def filter_df_by_date_time(self):
        
        self.filtered_df = self.original_df.copy(deep=True)

        self.filtered_df["seen"] = pd.to_datetime(self.filtered_df['seen'])

        self.filtered_df = self.filtered_df[self.filtered_df["seen"].between(self.StartDatetimePicker.value,self.EndDatetimePicker.value)]

    
    def plot_A(self):
        
        grouped = self.filtered_df.groupby(by=self.field_Select.value)['amount'].sum().reset_index()

        plt.bar(grouped[self.field_Select.value], grouped['amount'])

        plt.show()

        
    def plot_B(self):
        grouped = self.filtered_df.groupby(by=self.field_Select.value)['amount'].sum().reset_index()

        p = figure(
            x_range=grouped[self.field_Select.value],
            height=800,
                        width=800,
            tools="pan,wheel_zoom,box_zoom,reset,save"
        )

        source = ColumnDataSource(grouped)
        
        p.vbar(
            x=self.field_Select.value,
            top='amount',
            width=1,
            source=source
        )

        save(p)

        display(IFrame("bokeh_plot.html", width=600, height=1000))
        
    def plot_C(self):

        None
        
    def display_message_to_user(self,text_to_display):
        """
        Display the complete dashboard in the notebook.
        """
        self.warning_flag = True
        
        display(widgets.HTML(f"<p style='color:red;'>{text_to_display}</p>"))


    # --- Function to filter DataFrame based on filter descriptions ---
    def apply_user_filters(self):
        
        for btn in  self.filters_hbox.children:
            desc = btn.description.strip()
            
            if " NOT IN " in desc:
                col, val_str = desc.split(" NOT IN ", 1)
                values = ast.literal_eval(val_str)
                self.filtered_df = self.filtered_df[~self.filtered_df[col].isin(values)]
    
            elif " IN " in desc:
                col, val_str = desc.split(" IN ", 1)
                values = ast.literal_eval(val_str)  # safely convert string to list
                self.filtered_df = self.filtered_df[self.filtered_df[col].isin(values)]


    
    def get_feilds_to_plot(self,df):
        """
        Display the complete dashboard in the notebook.
        """
        to_remove = ['amount', 'seen']
        
        return  [x for x in list(self.original_df.columns) if x not in to_remove] 

    def filter_options(self):
        """
        genereates more complex widget filtering options
        """ 

        not_include_in_filter = ['amount', 'seen']

        #get only those that are a cetegory

        to_filter = [x for x in self.original_df.select_dtypes(include=['object', 'category']).columns.tolist() if x not in not_include_in_filter]

        filter_options_output = widgets.Output()
        filter_output = widgets.Output()    # For the SelectMultiple widget
        filter_button_output = widgets.Output()    # For displaying active filter buttons

    # --- Dropdown to pick column ---
        col_dropdown = widgets.Dropdown(
            options=to_filter,
            description="Column:"
        )

        # --- Function to create a multi-select widget for a column ---
        def create_filter_widget(col):
            return widgets.SelectMultiple(
                options=self.original_df[col].unique(),
                value=list(self.original_df[col].unique()),
                description=col
            )
        
        # --- Update filter widget when column changes ---
        def update_filter_widget(change):
            with filter_output:
                filter_output.clear_output()
                widget = create_filter_widget(change['new'])
                display(widget)
                filter_output.widget = widget  # store reference
    
         # Observe changes in dropdown
        col_dropdown.observe(update_filter_widget, names='value')   

        # --- Action Buttons ---
        include_value_btn = widgets.Button(description="Keep rows matching values",
                                           layout=widgets.Layout(width='auto'),
                                           button_style='success')
        exclude_value_btn = widgets.Button(description="Remove rows matching values",
                                           layout=widgets.Layout(width='auto'),
                                           button_style='danger')
        reset_btn = widgets.Button(description="Reset",
                                   layout=widgets.Layout(width='auto'),
                                   button_style='warning')

        Buttons_together = HBox([include_value_btn, exclude_value_btn, reset_btn])

        # --- VBox to hold filter buttons ---
        self.filters_hbox = widgets.HBox(    layout=widgets.Layout(
        flex_flow='row wrap',   
        gap='5px'
    ))

        # --- Helper function to find a button by substring in description ---
        def find_button(substring):
            return next((b for b in self.filters_hbox.children if substring in b.description), None)
        
        # --- Function to add/update include filter ---
        def add_include_filter(btn):
            desc = f"{col_dropdown.value} IN {list(filter_output.widget.value)}"
            existing = find_button(f"{col_dropdown.value} IN")
            
            if existing:
                existing.description = desc
                existing.layout.width = 'auto'  # adjust width
            else:
                new_btn = widgets.Button(description=desc,
                                         button_style='success',
                                         disabled=True,
                                         layout=widgets.Layout(width='auto'))
                self.filters_hbox.children = list(self.filters_hbox.children) + [new_btn]

        # --- Function to add/update exclude filter ---
        def add_exclude_filter(btn):
            desc = f"{col_dropdown.value} NOT IN {list(filter_output.widget.value)}"
            existing = find_button(f"{col_dropdown.value} NOT IN")
            
            if existing:
                existing.description = desc
                existing.layout.width = 'auto'
            else:
                new_btn = widgets.Button(description=desc,
                                         button_style='danger',
                                         disabled=True,
                                         layout=widgets.Layout(width='auto'))
                self.filters_hbox.children = list(self.filters_hbox.children) + [new_btn]
        
        # --- Reset function ---
        def reset_filters(btn):
            self.filters_hbox.children = []      # clear all filter buttons
            with filter_output:
                filter_output.clear_output()
                # recreate the filter widget for current column
                widget = create_filter_widget(col_dropdown.value)
                display(widget)
                filter_output.widget = widget

        # --- Connect buttons ---
        include_value_btn.on_click(add_include_filter)
        exclude_value_btn.on_click(add_exclude_filter)
        reset_btn.on_click(reset_filters)
        
        # # --- Display layout ---
        with filter_options_output:
            display(VBox([col_dropdown, filter_output, Buttons_together,widgets.HTML("<hr style='border:1px solid gray; width:100%'>"), self.filters_hbox]))
        
        # --- Initialize first filter widget ---
        update_filter_widget({'new': col_dropdown.value})

        return filter_options_output
        
    def show(self):
        """
        Display the complete dashboard in the notebook.
        """
        display(self.ui)

In [4]:
import pandas as pd

# Example DataFrame
df = pd.read_csv('test.csv')
# Create dashboard
dash = Dashboard(df)

# Show it
dash.show()